# Factor analysis and PPCA fitted with EM

Factor analysis (PML book 1, section 20.2) models $D$-dimensional data with $L < D$ latent factors:

$$p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I}), \qquad p(\mathbf{x} \mid \mathbf{z}) = \mathcal{N}(\mathbf{W}\mathbf{z} + \boldsymbol{\mu}, \boldsymbol{\Psi}),$$

with a diagonal noise covariance $\boldsymbol{\Psi}$, so $p(\mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \mathbf{W}\mathbf{W}^\top + \boldsymbol{\Psi})$.
PPCA is the special case $\boldsymbol{\Psi} = \sigma^2 \mathbf{I}$.

This notebook fits both models with the same EM algorithm on synthetic data whose noise variance differs per dimension,
checks FA against `sklearn.decomposition.FactorAnalysis` and PPCA against its closed-form MLE, and compares the two.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from sklearn.decomposition import FactorAnalysis

try:
    import probml_utils as pml
except ModuleNotFoundError:
    %pip install -qq git+https://github.com/probml/probml-utils.git
    import probml_utils as pml

## Data

$N = 500$ points in $D = 6$ dimensions from $L = 2$ factors; the noise variance grows from 0.05 to 2 across the dimensions.

In [ ]:
np.random.seed(0)

N, D, L = 500, 6, 2
W_true = np.random.randn(D, L)
psi_true = np.array([0.05, 0.1, 0.2, 0.5, 1.0, 2.0])  # a different noise variance per dimension
mu_true = np.random.randn(D)
Z = np.random.randn(N, L)
X = Z @ W_true.T + mu_true + np.random.randn(N, D) * np.sqrt(psi_true)

## EM

With $\tilde{\mathbf{x}}_n = \mathbf{x}_n - \boldsymbol{\mu}$ ($\boldsymbol{\mu}$ is the sample mean, its MLE) and $\mathbf{S}$ the sample covariance:

- **E step:** $p(\mathbf{z}_n \mid \mathbf{x}_n) = \mathcal{N}(\mathbf{m}_n, \mathbf{G})$, with
  $\mathbf{G} = (\mathbf{I} + \mathbf{W}^\top \boldsymbol{\Psi}^{-1} \mathbf{W})^{-1}$ and
  $\mathbf{m}_n = \mathbf{G} \mathbf{W}^\top \boldsymbol{\Psi}^{-1} \tilde{\mathbf{x}}_n$;
  $\mathbb{E}[\mathbf{z}_n \mathbf{z}_n^\top] = \mathbf{G} + \mathbf{m}_n \mathbf{m}_n^\top$.
- **M step:** $\mathbf{W} = \left(\sum_n \tilde{\mathbf{x}}_n \mathbf{m}_n^\top\right) \left(\sum_n \mathbb{E}[\mathbf{z}_n \mathbf{z}_n^\top]\right)^{-1}$, then
  - FA: $\boldsymbol{\Psi} = \mathrm{diag}\left(\mathbf{S} - \mathbf{W} \frac{1}{N} \sum_n \mathbf{m}_n \tilde{\mathbf{x}}_n^\top\right)$;
  - PPCA: $\sigma^2 = \frac{1}{D} \mathrm{tr}\left(\mathbf{S} - \mathbf{W} \frac{1}{N} \sum_n \mathbf{m}_n \tilde{\mathbf{x}}_n^\top\right)$, the average of the FA update.

The E step and the update for $\mathbf{W}$ are the same for both models, so one function fits both (`isotropic=True` for PPCA).

In [ ]:
def fa_loglik(X, mu, W, psi):
    C = W @ W.T + np.diag(psi)
    return multivariate_normal(mu, C).logpdf(X).sum()


def fa_em(X, L, isotropic=False, n_iter=5000, tol=1e-8, seed=0):
    """EM for factor analysis; isotropic=True ties the noise variances (PPCA)."""
    N, D = X.shape
    mu = X.mean(axis=0)
    Xc = X - mu
    S = Xc.T @ Xc / N
    rng = np.random.default_rng(seed)
    W = rng.standard_normal((D, L))
    psi = np.diag(S).copy()
    if isotropic:
        psi = np.full(D, psi.mean())
    lls = []
    for it in range(n_iter):
        # E step: the posterior covariance G is the same for every n
        WtPinv = W.T / psi  # W^T Psi^{-1}, [L, D]
        G = np.linalg.inv(np.eye(L) + WtPinv @ W)
        M = Xc @ (G @ WtPinv).T  # row n is m_n = E[z_n], [N, L]
        Ezz = N * G + M.T @ M  # sum_n E[z_n z_n^T]
        # M step
        W = (Xc.T @ M) @ np.linalg.inv(Ezz)
        psi = np.diag(S - W @ (M.T @ Xc) / N).copy()
        if isotropic:
            psi = np.full(D, psi.mean())  # sigma^2
        lls.append(fa_loglik(X, mu, W, psi))
        if it > 0 and abs(lls[-1] - lls[-2]) < tol * abs(lls[-2]):
            break
    return mu, W, psi, np.array(lls)


mu, W, psi, lls = fa_em(X, L)
_, W_ppca, psi_ppca, lls_ppca = fa_em(X, L, isotropic=True)
sigma2 = psi_ppca[0]
for name, ll in [("FA", lls), ("PPCA", lls_ppca)]:
    assert np.all(np.diff(ll) > -1e-6), f"{name} EM log-likelihood decreased"
    print(f"{name} EM: {len(ll)} iterations, log-likelihood {ll[-1]:.3f}")

## Checks

FA against scikit-learn, PPCA against its closed-form MLE (Tipping and Bishop):
$\sigma^2$ is the mean of the $D - L$ smallest eigenvalues of $\mathbf{S}$, and $\mathbf{W} = \mathbf{U}_L (\boldsymbol{\Lambda}_L - \sigma^2 \mathbf{I})^{1/2}$.
$\mathbf{W}$ is identified only up to a rotation of the latent space, so loadings are compared through $\mathbf{W}\mathbf{W}^\top$.

In [ ]:
fa = FactorAnalysis(n_components=L, tol=1e-8, max_iter=10000).fit(X)
W_sk = fa.components_.T
ll_sk = fa.score(X) * N
print(f"FA, sklearn: log-likelihood {ll_sk:.3f}")
print(f"  max |W W^T - W_sk W_sk^T| = {np.abs(W @ W.T - W_sk @ W_sk.T).max():.3f}")
print(f"  max |W W^T - W_true W_true^T| = {np.abs(W @ W.T - W_true @ W_true.T).max():.3f}")

evals, evecs = np.linalg.eigh(np.cov(X.T, bias=True))
evals, evecs = evals[::-1], evecs[:, ::-1]
sigma2_cf = evals[L:].mean()
W_cf = evecs[:, :L] * np.sqrt(evals[:L] - sigma2_cf)
ll_cf = fa_loglik(X, mu, W_cf, np.full(D, sigma2_cf))
print(f"PPCA, closed form: sigma^2 {sigma2_cf:.4f} (EM {sigma2:.4f}), log-likelihood {ll_cf:.3f}")
print(f"  max |W W^T - W_cf W_cf^T| = {np.abs(W_ppca @ W_ppca.T - W_cf @ W_cf.T).max():.4f}")

In [ ]:
colors = ["#2a78d6", "#eb6834", "#1baf7a"]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(lls) + 1), lls, color=colors[0], lw=2, label="FA, EM")
ax.plot(np.arange(1, len(lls_ppca) + 1), lls_ppca, color=colors[1], lw=2, label="PPCA, EM")
ax.axhline(ll_sk, color=colors[2], ls="--", lw=1.5, label="FA, sklearn")
ax.axhline(ll_cf, color="0.4", ls=":", lw=1.5, label="PPCA, closed form")
ax.set_xscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel("log-likelihood")
ax.legend()
fig.tight_layout()
pml.savefig("fa_em_loglik.pdf")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(D)
w = 0.2
estimates = [  # each method keeps its color from the log-likelihood plot
    (psi_true, r"true $\Psi$", "0.6"),
    (psi, "FA, EM", colors[0]),
    (fa.noise_variance_, "FA, sklearn", colors[2]),
    (psi_ppca, r"PPCA, EM $\sigma^2$", colors[1]),
]
for i, (vals, name, color) in enumerate(estimates):
    ax.bar(x + (i - 1.5) * w, vals, w * 0.9, color=color, label=name)
ax.set_xticks(x)
ax.set_xticklabels([f"$x_{j + 1}$" for j in range(D)])
ax.set_ylabel("noise variance")
ax.legend()
fig.tight_layout()
pml.savefig("fa_em_noise.pdf")
plt.show()

Both EM fits reach their reference solutions: FA matches scikit-learn (log-likelihoods within 0.02), and PPCA matches
the closed form ($\sigma^2$ to 4 decimals, the same log-likelihood).

- **Fit:** FA recovers the per-dimension noise variances; PPCA has to use a single $\sigma^2$ for all of them, so it
  overestimates the noise in the clean dimensions, underestimates it in the noisy ones, and its log-likelihood is
  about 380 lower.
- **Convergence:** PPCA's EM converges in about 40 iterations, FA's takes about 700. Both gain most of the
  log-likelihood in the first 10 iterations; FA then has a long slow tail while the six noise variances settle.
- **When PPCA is enough:** with the same noise level in every dimension the two models coincide, and PPCA also has a
  closed-form solution, so EM for PPCA is mainly useful for large or incomplete data: written without $\mathbf{S}$,
  each iteration costs $O(NDL)$ instead of an eigendecomposition of a $D \times D$ matrix.